In [4]:
import pandas as pd
import json

# -----------------------------
# 1. metric 결과 로드
# -----------------------------
embedding_metric_df = pd.read_csv(
    "/home/ubuntu/work/dahyeon/evaluation/dataset/embedding_metric_results.csv"
)

# -----------------------------
# 2. scenario 파일 로드
# -----------------------------
with open(
    "/home/ubuntu/work/dahyeon/evaluation/dataset/scenario_data_after_retrieval.json",
    "r",
    encoding="utf-8"
) as f:
    scenario_data = json.load(f)

# -----------------------------
# 3. scenario_id → query_type 매핑
# -----------------------------
query_type_map = {
    item["scenario_id"]: item["query_type"]
    for item in scenario_data
}

# -----------------------------
# 4. query_type 컬럼 추가
# -----------------------------
embedding_metric_df["query_type"] = (
    embedding_metric_df["query_id"]
    .map(query_type_map)
)

# 확인
print(
    embedding_metric_df[
        ["query_id", "embedding_model", "query_type"]
    ].head(20)
)

   query_id embedding_model        query_type
0       S01       bge_m3_ko      anchor_based
1       S01           clova      anchor_based
2       S01            kure      anchor_based
3       S02       bge_m3_ko      anchor_based
4       S02           clova      anchor_based
5       S02            kure      anchor_based
6       S03       bge_m3_ko      anchor_based
7       S03           clova      anchor_based
8       S03            kure      anchor_based
9       S04       bge_m3_ko     topic_purpose
10      S04           clova     topic_purpose
11      S04            kure     topic_purpose
12      S05       bge_m3_ko     topic_purpose
13      S05           clova     topic_purpose
14      S05            kure     topic_purpose
15      S06       bge_m3_ko     topic_purpose
16      S06           clova     topic_purpose
17      S06            kure     topic_purpose
18      S07       bge_m3_ko  topic_constraint
19      S07           clova  topic_constraint


In [5]:
type_result = (
    embedding_metric_df
    .groupby(
        ["query_type", "embedding_model"]
    )["recall"]
    .mean()
    .round(4)
    .reset_index()
)

print(type_result)

            query_type embedding_model  recall
0         anchor_based       bge_m3_ko  0.5000
1         anchor_based           clova  0.5000
2         anchor_based            kure  0.5000
3   availability_first       bge_m3_ko  0.3108
4   availability_first           clova  0.2634
5   availability_first            kure  0.2661
6      broad_ambiguous       bge_m3_ko  0.1273
7      broad_ambiguous           clova  0.3333
8      broad_ambiguous            kure  0.2404
9      pure_mood_state       bge_m3_ko  0.2222
10     pure_mood_state           clova  0.2222
11     pure_mood_state            kure  0.1667
12          pure_topic       bge_m3_ko  0.4206
13          pure_topic           clova  0.3463
14          pure_topic            kure  0.4979
15    topic_constraint       bge_m3_ko  0.3064
16    topic_constraint           clova  0.3670
17    topic_constraint            kure  0.3805
18       topic_purpose       bge_m3_ko  0.3300
19       topic_purpose           clova  0.6483
20       topi

In [6]:
winner_df = (
    embedding_metric_df
    .sort_values(
        ["query_id", "recall"],
        ascending=[True, False]
    )
    .groupby("query_id")
    .first()
    .reset_index()
)

winner_df = winner_df[
    [
        "query_id",
        "embedding_model",
        "recall",
        "mrr",
        "ndcg"
    ]
]

winner_df

,query_id,embedding_model,recall,mrr,ndcg
0,S01,bge_m3_ko,0.166667,0.200000,0.117063
1,S02,bge_m3_ko,0.333333,0.100000,0.135652
2,S03,bge_m3_ko,1.000000,0.500000,0.591235
3,S04,kure,0.875000,0.500000,0.601111
4,S05,clova,1.000000,0.142857,0.381622
5,S06,kure,0.360000,1.000000,0.555838
6,S07,clova,0.545455,1.000000,0.564657
7,S08,clova,0.222222,0.125000,0.144904
8,S09,kure,0.555556,1.000000,0.623073
9,S10,clova,0.384615,0.500000,0.494117


# 청킹 전략 결과 분석

In [10]:
import json
import pandas as pd
from pathlib import Path

# =====================================================
# 1. 파일 경로
# =====================================================

METRIC_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/chunk_metric_results.csv")
SCENARIO_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/scenario_data_after_retrieval.json")
SAVE_DIR = Path("/home/ubuntu/work/dahyeon/evaluation/dataset")

# =====================================================
# 2. 데이터 로드
# =====================================================

metric_df = pd.read_csv(METRIC_PATH)

with SCENARIO_PATH.open("r", encoding="utf-8") as f:
    scenario_data = json.load(f)

# =====================================================
# 3. query_type 추가
# =====================================================

query_type_map = {
    item["scenario_id"]: item["query_type"]
    for item in scenario_data
}

metric_df["query_type"] = metric_df["query_id"].map(query_type_map)

# =====================================================
# 4. 전체 평균 비교
# =====================================================

overall_result = (
    metric_df
    .groupby("retriever")[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
    .reset_index()
)

print("=== 전체 평균 성능 ===")
print(overall_result)

# =====================================================
# 5. query_type별 성능 비교
# =====================================================

type_result = (
    metric_df
    .groupby(["query_type", "retriever"])[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
    .reset_index()
)

print("\n=== Query Type별 성능 ===")
print(type_result)

# =====================================================
# 6. query_type별 full/chunk recall 차이
# =====================================================

type_recall_pivot = type_result.pivot(
    index="query_type",
    columns="retriever",
    values="recall"
)

type_recall_pivot["chunk_minus_full"] = (
    type_recall_pivot["chunk"] - type_recall_pivot["full"]
)

type_recall_pivot = type_recall_pivot.sort_values(
    "chunk_minus_full",
    ascending=False
)

print("\n=== Query Type별 Recall 차이 ===")
print(type_recall_pivot)

# =====================================================
# 7. query별 full/chunk recall 차이
# =====================================================

query_recall_pivot = metric_df.pivot_table(
    index="query_id",
    columns="retriever",
    values="recall"
).reset_index()

query_recall_pivot["chunk_minus_full"] = (
    query_recall_pivot["chunk"] - query_recall_pivot["full"]
)

query_recall_pivot = query_recall_pivot.sort_values(
    "chunk_minus_full",
    ascending=False
)

print("\n=== Query별 Recall 차이 ===")
print(query_recall_pivot)

# # =====================================================
# # 8. 결과 저장
# # =====================================================

# metric_df.to_csv(
#     SAVE_DIR / "chunk_metric_with_query_type.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# overall_result.to_csv(
#     SAVE_DIR / "chunk_overall_result.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# type_result.to_csv(
#     SAVE_DIR / "chunk_query_type_result.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# type_recall_pivot.to_csv(
#     SAVE_DIR / "chunk_query_type_recall_diff.csv",
#     encoding="utf-8-sig"
# )

# query_recall_pivot.to_csv(
#     SAVE_DIR / "chunk_query_recall_diff.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# print("\n저장 완료")

=== 전체 평균 성능 ===
  retriever     hit  recall     mrr    ndcg
0     chunk  0.8571  0.2693  0.3499  0.2533
1      full  0.9524  0.3995  0.5325  0.4005

=== Query Type별 성능 ===
            query_type retriever     hit  recall     mrr    ndcg
0         anchor_based     chunk  0.6667  0.3889  0.1278  0.2132
1         anchor_based      full  1.0000  0.5000  0.4722  0.3851
2   availability_first     chunk  1.0000  0.2131  0.3254  0.3168
3   availability_first      full  1.0000  0.2661  0.7778  0.4946
4      broad_ambiguous     chunk  1.0000  0.1172  0.1690  0.1060
5      broad_ambiguous      full  1.0000  0.2404  0.4405  0.2721
6      pure_mood_state     chunk  0.6667  0.1667  0.2143  0.1505
7      pure_mood_state      full  0.6667  0.1667  0.3590  0.1518
8           pure_topic     chunk  0.6667  0.1559  0.6667  0.3079
9           pure_topic      full  1.0000  0.4979  0.6111  0.6023
10    topic_constraint     chunk  1.0000  0.3064  0.4833  0.2910
11    topic_constraint      full  1.0000  0.380